# School-site suitability mapping: TA Chimaliro, Thyolo

This notebook downloads data, prepares layers, applies the assignment criteria and creates static/interactive maps. **Verify the boundary name and district before accepting any result.**

In [2]:
import geopandas as gpd
import pandas as pd
import matplotlib.pyplot as plt
import folium
import requests
import os
import warnings
warnings.filterwarnings('ignore')

# Define paths
BASE_DIR = os.path.abspath("..")
DATA_DIR = os.path.join(BASE_DIR, "data")
OUTPUT_DIR = os.path.join(BASE_DIR, "outputs")

print("Setup complete. Working directories established.")

Setup complete. Working directories established.


## 1. Retrieve and verify the TA boundary
The queried ArcGIS layer is a source lead, not automatic proof of the correct TA. Review fields and matching records. If the feature is not TA Chimaliro in Thyolo, replace it with a verified TA boundary and document the replacement.

In [5]:
datasets = {
    "ta": "https://ndatarepo.must.ac.mw/layers/malawi-ta-traditional-authority/geojson/", 
    "protected": "https://ndatarepo.must.ac.mw/layers/malawi-protected-areas/geojson/",
    "schools": "https://ndatarepo.must.ac.mw/layers/malawi-primary-schools/geojson/"
}
# 2. Fetch the list of all datasets from your GeoPortal
print("Connecting to MUST GeoPortal...")
try:
    response = requests.get("http://ndatarepo.must.ac.mw/layers/")
    response.raise_for_status()
    data = response.json()
    datasets = data.get("datasets", [])
    print(f"✅ Found {len(datasets)} datasets on the portal.")
except Exception as e:
    print(f"❌ Failed to connect to portal: {e}")
    datasets = []

# 3. Automatically find the IDs and GeoJSON URLs for our target datasets
target_files = {
    "ta": None,
    "protected": None,
    "schools": None
}

for ds in datasets:
    title = ds["title"].lower()
    geojson_url = ds.get("geojson_url")
    
    if not geojson_url:
        continue

    if "traditional authority" in title or " ta" in title:
        target_files["ta"] = geojson_url
    elif "protected" in title:
        target_files["protected"] = geojson_url
    elif "school" in title:
        target_files["schools"] = geojson_url

# 4. Download the GeoJSONs
for name, url in target_files.items():
    if not url:
        print(f"⚠️ Could not find a URL for {name}.")
        continue
        
    print(f"Downloading {name} from {url}...")
    try:
        # We append ?format=geojson or just read it directly since the endpoint returns GeoJSON
        gdf = gpd.read_file(url)
        gdf.to_file(os.path.join(DATA_DIR, f"{name}.geojson"), driver="GeoJSON")
        print(f"✅ Successfully downloaded {name}: {len(gdf)} features.")
    except Exception as e:
        print(f"❌ Failed to download {name}: {e}")

# 5. Load local roads
local_roads_path = r"C:\Users\YOUR_USERNAME\Downloads\malawi_roads.shp" # <-- CHANGE THIS PATH
try:
    roads = gpd.read_file(local_roads_path)
    roads.to_file(os.path.join(DATA_DIR, "roads.geojson"), driver="GeoJSON")
    print(f"✅ Successfully loaded local roads: {len(roads)} features.")
except Exception as e:
    print(f"❌ Failed to load roads. Check the path! Error: {e}")

for name, url in datasets.items():
    print(f"Downloading {name}...")
    try:
        gdf = gpd.read_file(url)
        gdf.to_file(os.path.join(DATA_DIR, f"{name}.geojson"), driver="GeoJSON")
        print(f"✅ Successfully downloaded {name}: {len(gdf)} features.")
    except Exception as e:
        print(f"❌ Failed to download {name}: {e}")

# Load your local Malawi Roads shapefile (Update the path!)
local_roads_path = r"D:\DATA\Mw_topographic_data\mw_roads.shp"
try:
    roads = gpd.read_file(local_roads_path)
    roads.to_file(os.path.join(DATA_DIR, "roads.geojson"), driver="GeoJSON")
    print(f"✅ Successfully loaded local roads: {len(roads)} features.")
except Exception as e:
    print(f"❌ Failed to load roads. Check the path! Error: {e}")

Connecting to MUST GeoPortal...
❌ Failed to connect to portal: Expecting value: line 2 column 1 (char 1)
⚠️ Could not find a URL for ta.
⚠️ Could not find a URL for protected.
⚠️ Could not find a URL for schools.
❌ Failed to load roads. Check the path! Error: C:\Users\YOUR_USERNAME\Downloads\malawi_roads.shp: No such file or directory


AttributeError: 'list' object has no attribute 'items'

### Select the Chimaliro feature
The selection below is deliberately strict. Inspect the candidate row and verify its district. If there is no unambiguous match, stop and use a verified boundary dataset.

In [ ]:
name_fields=[c for c in bounds.columns if any(k in c.lower() for k in ("name","admin","ta","district"))]
print("Possible name fields:",name_fields)
for c in name_fields:
    hit=bounds[bounds[c].astype(str).str.contains("Chimaliro",case=False,na=False)]
    if not hit.empty: print("Matches in",c); display(hit.drop(columns="geometry"))
name_field=next((c for c in name_fields if bounds[c].astype(str).str.contains("Chimaliro",case=False,na=False).any()),None)
if name_field is None: raise RuntimeError("Chimaliro not found. Load a verified TA boundary rather than guessing.")
ta=bounds[bounds[name_field].astype(str).str.contains("Chimaliro",case=False,na=False)].copy()
if len(ta)!=1: display(ta.drop(columns="geometry")); raise RuntimeError("Ambiguous boundary. Filter by district and verify manually.")
ta=ta.to_crs(CRS); ta["geometry"]=ta.geometry.make_valid(); ta=ta[~ta.geometry.is_empty]
ta.to_file(RAW/"ta_chimaliro_boundary.geojson",driver="GeoJSON")
print("Boundary area km²:",round(ta.geometry.area.sum()/1e6,3)); display(ta.drop(columns="geometry"))

## 2. Retrieve roads, schools and protected areas from OpenStreetMap
Road classes are defined here as motorway, trunk, primary, secondary and tertiary, including links. School features are filtered for primary-school tags. Protected-area tags in OSM may be incomplete and must be checked against an authoritative inventory where possible.

In [ ]:
poly=ta.to_crs("EPSG:4326").geometry.union_all()
def osm(tags,label):
    try:
        g=ox.features_from_polygon(poly,tags)
        if g.empty: print("WARNING: no",label,"features"); return gpd.GeoDataFrame(geometry=[],crs="EPSG:4326")
        g=g.reset_index(); print(label, len(g)); return g
    except Exception as e:
        print("Query failed for",label,e); return gpd.GeoDataFrame(geometry=[],crs="EPSG:4326")
roads0=osm({"highway":["motorway","trunk","primary","secondary","tertiary","motorway_link","trunk_link","primary_link","secondary_link","tertiary_link"]},"roads")
schools0=osm({"amenity":"school"},"schools")
protected0=osm({"boundary":["protected_area","national_park"],"leisure":"nature_reserve"},"protected areas")
if roads0.empty or schools0.empty: raise RuntimeError("Missing roads or schools. Use verified local layers or retry data download.")
roads=roads0.to_crs(CRS); schools=schools0.to_crs(CRS); protected=protected0.to_crs(CRS) if not protected0.empty else protected0
def clean_clip(g,polygon):
    if g.empty: return g
    g=g[g.geometry.notna() & ~g.geometry.is_empty].copy(); g["geometry"]=g.geometry.make_valid(); g=g[~g.geometry.is_empty]
    return gpd.clip(g,polygon)
ta_geom=ta.geometry.union_all(); roads=clean_clip(roads,ta_geom); schools=clean_clip(schools,ta_geom)
if not protected.empty: protected=clean_clip(protected,ta_geom)
def primary(row):
    txt=" ".join(str(row.get(c,"")) for c in ("name","school","isced:level","grades","description")).lower()
    return "primary" in txt or str(row.get("isced:level","")) in {"1","1;2"}
primary_schools=schools[schools.apply(primary,axis=1)].copy()
if primary_schools.empty: display(schools.drop(columns="geometry").head(25)); raise RuntimeError("No primary schools identified by tags. Review records and document a defensible filter.")
print("Roads:",len(roads),"Primary-school candidates:",len(primary_schools),"Protected-area features:",len(protected))

## 3. Record data metadata

In [ ]:
metadata=pd.DataFrame([
 {"layer":"TA boundary","source":url,"retrieved_utc":retrieved,"source_format":"ArcGIS REST GeoJSON","source_crs":"EPSG:4326","analysis_crs":CRS,"limitations":"Verify exact TA and district; source vintage may vary."},
 {"layer":"Roads","source":"OpenStreetMap via OSMnx","retrieved_utc":retrieved,"source_format":"OSM features","source_crs":"EPSG:4326","analysis_crs":CRS,"limitations":"Coverage/classification/position vary."},
 {"layer":"Primary schools","source":"OpenStreetMap via OSMnx","retrieved_utc":retrieved,"source_format":"OSM features","source_crs":"EPSG:4326","analysis_crs":CRS,"limitations":"Primary-level tagging and completeness need verification."},
 {"layer":"Protected areas","source":"OpenStreetMap via OSMnx","retrieved_utc":retrieved,"source_format":"OSM features","source_crs":"EPSG:4326","analysis_crs":CRS,"limitations":"May omit protected areas; compare official inventory."}])
metadata.to_csv(PROC/"data_sources_metadata.csv",index=False); display(metadata)

## 4. Apply the three criteria
Exclude areas within 2,000 m of existing primary schools, retain areas within 500 m of main roads, subtract mapped protected areas, and clip to the TA boundary. Distances are in metres because all layers are projected to EPSG:32736.

In [ ]:
school_exclusion=primary_schools.geometry.buffer(2000).union_all()
road_access=roads.geometry.buffer(500).union_all()
suitable=ta_geom.intersection(road_access).difference(school_exclusion)
if not protected.empty: suitable=suitable.difference(protected.geometry.union_all())
suitable=suitable.intersection(ta_geom).make_valid()
suitable_gdf=gpd.GeoDataFrame({"criterion":["within 500m of main roads; at least 2km from mapped primary schools; outside mapped protected areas"]},geometry=[suitable],crs=CRS)
suitable_gdf=suitable_gdf[~suitable_gdf.geometry.is_empty].copy(); suitable_gdf["area_km2"]=suitable_gdf.geometry.area/1e6
area_km2=float(suitable_gdf.area_km2.sum()); print("Computed suitable area (km²):",round(area_km2,3))
suitable_gdf.to_file(OUT/"suitable_areas.gpkg",layer="suitable_areas",driver="GPKG"); display(suitable_gdf[["criterion","area_km2"]])

## 5. Static map

In [ ]:
fig,ax=plt.subplots(figsize=(10,10))
ta.boundary.plot(ax=ax,color="black",linewidth=1.4,label="TA boundary")
if not protected.empty: protected.plot(ax=ax,color="#7ca982",edgecolor="#356b3d",alpha=.6,label="Mapped protected areas")
suitable_gdf.plot(ax=ax,color="#f4d35e",edgecolor="#9a7400",linewidth=.3,label="Suitable areas")
roads.plot(ax=ax,color="#b64b3b",linewidth=.6,label="Main roads")
primary_schools.plot(ax=ax,color="#2f5d9b",markersize=10,marker="^",label="Existing primary schools")
ax.set_title("Potential Primary School Suitability — TA Chimaliro, Thyolo"); ax.set_axis_off()
ax.annotate("N",xy=(.93,.92),xytext=(.93,.82),xycoords="axes fraction",ha="center",fontweight="bold",arrowprops=dict(arrowstyle="-|>",lw=1.8))
xmin,ymin,xmax,ymax=ta.total_bounds; x0=xmin+(xmax-xmin)*.06; y0=ymin+(ymax-ymin)*.05
ax.plot([x0,x0+2000],[y0,y0],color="black",linewidth=3); ax.text(x0+1000,y0+(ymax-ymin)*.015,"2 km",ha="center",fontsize=9)
ax.legend(loc="lower left",fontsize=8); fig.text(.01,.01,f"Sources: TA boundary service; OpenStreetMap. Retrieved {retrieved[:10]}. CRS: {CRS}. Screening analysis; field validation required.",fontsize=7)
fig.tight_layout(rect=[0,.035,1,1]); fig.savefig(OUT/"suitable_areas_map.png",dpi=300,bbox_inches="tight"); plt.show()

## 6. Interactive map with layer controls and popups

In [ ]:
ta_ll=ta.to_crs("EPSG:4326"); roads_ll=roads.to_crs("EPSG:4326"); schools_ll=primary_schools.to_crs("EPSG:4326"); suitable_ll=suitable_gdf.to_crs("EPSG:4326")
protected_ll=protected.to_crs("EPSG:4326") if not protected.empty else protected
cent=ta_ll.geometry.union_all().centroid; m=folium.Map(location=[cent.y,cent.x],zoom_start=12,tiles="OpenStreetMap")
folium.GeoJson(ta_ll,name="TA boundary",style_function=lambda f:{"color":"black","weight":2,"fillOpacity":.02}).add_to(m)
folium.GeoJson(roads_ll,name="Main roads",style_function=lambda f:{"color":"#b64b3b","weight":2},tooltip=folium.GeoJsonTooltip(fields=["highway"]) if "highway" in roads_ll.columns else None).add_to(m)
fields=[c for c in ["name","amenity","school","isced:level"] if c in schools_ll.columns]
folium.GeoJson(schools_ll,name="Primary schools",marker=folium.CircleMarker(radius=4,color="#2f5d9b",fill=True),tooltip=folium.GeoJsonTooltip(fields=fields) if fields else None).add_to(m)
if not protected_ll.empty: folium.GeoJson(protected_ll,name="Mapped protected areas",style_function=lambda f:{"color":"#356b3d","fillColor":"#7ca982","fillOpacity":.4}).add_to(m)
folium.GeoJson(suitable_ll,name="Suitable areas",style_function=lambda f:{"color":"#9a7400","fillColor":"#f4d35e","fillOpacity":.65},tooltip=folium.GeoJsonTooltip(fields=["area_km2"])).add_to(m)
folium.LayerControl(collapsed=False).add_to(m); b=ta_ll.total_bounds; m.fit_bounds([[b[1],b[0]],[b[3],b[2]]]); m.save(str(OUT/"interactive_map.html")); display(m)

## 7. Results and limitations

In [ ]:
summary={"retrieved_utc":retrieved,"analysis_crs":CRS,"roads_features":len(roads),"primary_school_features":len(primary_schools),"protected_area_features":len(protected),"suitable_area_km2":round(area_km2,3),"criteria":{"school_min_distance_m":2000,"road_max_distance_m":500,"exclude_protected_areas":True}}
(OUT/"result_summary.json").write_text(json.dumps(summary,indent=2)); print(json.dumps(summary,indent=2))
print("Limitations: confirm TA/district; OSM layers may be incomplete; protected areas need official verification; distance buffers do not account for terrain, walking networks, land ownership, flood risk, population demand or school capacity. Candidate areas require field verification.")